# Episode 11: Build a Research Assistant

Research is inherently iterative — gather, draft, review, revise. That's exactly what a multi-agent pipeline does naturally.

In this episode, you'll build a 3-agent research pipeline that researches, writes, and reviews — with a feedback loop for revisions.

> **Keep in mind:** These agents use LLM knowledge to gather information. In Episode 12, we'll give them YOUR documents with RAG — making this pipeline much more powerful.

## Multi-stage research pattern

Real research isn't a single step. BetterFutureLabs found that collaborative agent discourse can "shave days off research and analysis" — and the key is iteration. The pipeline follows a cycle:

```
Researcher → Writer → Reviewer ──┐
                 ↑                  │
                 └── (needs revision) ┘
                       OR
                 └── (approved) → DONE
```

Each agent has a distinct role in that cycle:

1. **Researcher** gathers facts and key points
2. **Writer** drafts a structured report from those facts
3. **Reviewer** critiques the draft — either sending it back to the writer for revision, or approving it

This is a simplified version of the [GPT Researcher 8-agent architecture](https://ag2ai.github.io/ag2/blog/2026-03-03-GPT-Researcher-AG2), which uses an Editor → Researcher → Reviewer → Revisor pattern with multiple research sub-agents.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group.patterns import DefaultPattern
from autogen.agentchat.group import (
    AgentTarget,
    OnCondition,
    StringLLMCondition,
    TerminateTarget,
)

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")

## Create the research team

Now that you've seen the pattern, let's build the agents. Each one has a clear role and knows what to hand off and when. Let's see this:

In [ ]:
with llm_config:
    researcher = ConversableAgent(
        name="researcher",
        system_message=(
            "You are a research specialist. Given a topic, gather key facts, trends, "
            "statistics, and notable developments. Organize your findings as a structured "
            "list of bullet points grouped by subtopic.\n\n"
            "Be thorough but concise. Focus on the most important and recent information. "
            "When you have gathered enough material, hand off to the writer."
        ),
        human_input_mode="NEVER",
    )

    writer = ConversableAgent(
        name="writer",
        system_message=(
            "You are a report writer. Take the researcher's findings and write a clear, "
            "well-structured report with:\n"
            "- An executive summary (2-3 sentences)\n"
            "- Key sections with headers\n"
            "- A conclusion with forward-looking insights\n\n"
            "Write for a technical audience. Be precise and avoid filler. "
            "If the reviewer requests revisions, incorporate their feedback and produce "
            "an improved version.\n\n"
            "When your draft is ready, hand off to the reviewer."
        ),
        human_input_mode="NEVER",
    )

    reviewer = ConversableAgent(
        name="reviewer",
        system_message=(
            "You are a senior editor and reviewer. Evaluate the writer's report for:\n"
            "- Accuracy and completeness\n"
            "- Clarity and structure\n"
            "- Missing important points\n"
            "- Unsupported claims\n\n"
            "If the report needs improvement, provide specific, actionable feedback "
            "and request a revision. If the report is good enough, approve it by "
            "saying 'APPROVED' and providing a brief summary of the report's strengths."
        ),
        human_input_mode="NEVER",
    )

## Configure the handoff chain

This is where the feedback loop lives. The researcher always hands off to the writer, and the writer always hands off to the reviewer. But the reviewer has two options: send it back to the writer for revision, or terminate when the report is approved.

This leads to the core of the pattern — conditional routing at the review stage. Let's see this:

In [ ]:
# Researcher -> Writer (always)
researcher.set_after_work(AgentTarget(writer))

# Writer -> Reviewer (always)
writer.set_after_work(AgentTarget(reviewer))

# Reviewer -> Writer (if revision needed) OR terminate (if approved)
reviewer.handoffs = [
    OnCondition(
        target=AgentTarget(writer),
        condition=StringLLMCondition(
            prompt="The report needs revision. There are specific improvements to make."
        ),
    ),
    OnCondition(
        target=TerminateTarget(),
        condition=StringLLMCondition(
            prompt="The report is approved. It meets quality standards and is ready for delivery."
        ),
    ),
]

## Run the research pipeline

Now that you've seen the full handoff chain, let's watch the feedback loop in action. The researcher gathers facts, the writer drafts a report, and the reviewer either requests revisions or approves.

In [ ]:
pattern = DefaultPattern(
    initial_agent=researcher,
    agents=[researcher, writer, reviewer],
)

result, context_out, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="Research the current state of multi-agent AI systems in 2026",
    max_rounds=8,
)

## Understanding the feedback loop

Look at the output above. You should see a pattern emerge: the researcher produces bullet points, the writer drafts a structured report, and the reviewer critiques it with specific feedback.

Then the writer revises based on that feedback, and the reviewer approves the improved version. This feedback loop is what separates good multi-agent systems from simple chains — the reviewer acts as a quality gate, and the report keeps improving until it meets the bar.

The `max_rounds=8` parameter ensures the loop doesn't run forever. In practice, most reports are approved after 1-2 revision cycles.

## Print the final report

Let's extract the writer's last message to see the polished result.

In [ ]:
# Find the last message from the writer (the final report)
for message in reversed(result.messages):
    if message.get("name") == "writer":
        print("=" * 60)
        print("FINAL REPORT")
        print("=" * 60)
        print(message["content"])
        break

## Going deeper: production research systems

This 3-agent pipeline is a simplified version of what production research systems use. For more advanced implementations, explore these resources:

- **[GPT Researcher with AG2](https://ag2ai.github.io/ag2/blog/2026-03-03-GPT-Researcher-AG2)** — The full Editor → Researcher → Reviewer → Revisor pattern with multiple research sub-agents
- **`ag-ui/gpt-researcher/`** — A production GPT Researcher implementation with a web UI
- **`deep-research-agent/`** — A deep research agent that performs iterative web searches

These scale the pattern by adding more specialized agents (fact-checkers, source validators, citation formatters) and integrating web search tools for real-time information gathering.

## Try it yourself

What happens if the researcher's findings contain contradictory claims? Right now, the writer would just include both — and the reviewer might not catch the inconsistency.

Try adding a **fact-checker** agent between the researcher and the writer. The fact-checker should:

1. Receive the researcher's findings
2. Flag any claims that seem unsupported or contradictory
3. Hand off the validated findings to the writer

Update the handoff chain: `Researcher → Fact-Checker → Writer → Reviewer`

In [ ]:
# Your code here


## What's next?

Your research pipeline is powerful — but it only knows what the LLM was trained on. What if your agents could search your own company docs, policies, and data?